# Grade 10 · Unit 3 — Relationships and the line of best fit

> **SYNTHETIC** data.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def data_file(dataset, filename):
    """Find datasets/<dataset>/clean/<filename> by walking up from this folder."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        p = folder / "datasets" / dataset / "clean" / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename}. Open this notebook from inside the data-science-nepal folder.")

air = pd.read_csv(data_file("kathmandu-air-quality", "air_daily_synthetic.csv"), parse_dates=["date"])
air["pm25"] = air["pm25"].replace(-999, np.nan)     # -999 is a sensor error code, not a reading
print(air.shape, "| missing PM2.5:", int(air["pm25"].isna().sum()))

## Scatter: PM2.5 vs PM10
**Predict:** direction and strength?

In [ ]:
d = air.dropna(subset=["pm25"])
plt.scatter(d["pm25"], d["pm10"], s=8, alpha=0.6); plt.xlabel("PM2.5"); plt.ylabel("PM10"); plt.show()
r = d["pm25"].corr(d["pm10"])
print("correlation r =", round(r, 2))
assert -1 <= r <= 1

## Line of best fit by least squares

In [ ]:
slope, intercept = np.polyfit(d["pm25"], d["pm10"], 1)
print(f"PM10 ≈ {slope:.2f} × PM2.5 + {intercept:.1f}")
xs = np.linspace(d["pm25"].min(), d["pm25"].max(), 50)
plt.scatter(d["pm25"], d["pm10"], s=8, alpha=0.4); plt.plot(xs, slope * xs + intercept, color="red")
plt.show()

## Residuals — what the line misses

In [ ]:
resid = d["pm10"] - (slope * d["pm25"] + intercept)
print("mean residual ≈", round(resid.mean(), 6), "| typical miss (SD):", round(resid.std(), 1))
assert abs(resid.mean()) < 1e-6

## Correlation is not causation
Month drives both temperature and rain. Look at the climate table: does temperature correlate with rainfall? Could something else explain it?

In [ ]:
clim = pd.read_csv(data_file("climate-monthly", "climate_monthly_synthetic.csv"))
print("r(temp, rain) =", round(clim["mean_temp_c"].corr(clim["rain_mm"]), 2))
print(clim.groupby("month")[["mean_temp_c", "rain_mm"]].mean().round(1).T)

**Make:** write one sentence interpreting the slope in plain words, plus one reason correlation here is not causation.